# LoGeR Colab Demo

This notebook sets up LoGeR on Google Colab, downloads the released checkpoints, and runs the interactive `viser` demo with a shareable public URL.

- Use the sample sequence at `data/examples/office` by default.
- Switch `MODEL_VARIANT` or `INPUT_PATH` in the config cell to try other data.
- Keep the final cell running while you use the `viser` link.


In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/LoGeR')
if not REPO_DIR.exists():
    !git clone https://github.com/junyi42/LoGeR /content/LoGeR

%cd /content/LoGeR
!python -m pip install -q uv
!uv pip install --system -r requirements.txt onnxruntime pyyaml requests


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil

repo_id = 'Junyi42/LoGeR'
for model_name in ['LoGeR', 'LoGeR_star']:
    target_dir = Path('ckpts') / model_name
    target_dir.mkdir(parents=True, exist_ok=True)
    for filename in ['latest.pt', 'original_config.yaml']:
        target_path = target_dir / filename
        if not target_path.exists():
            downloaded = hf_hub_download(
                repo_id=repo_id,
                filename=f'{model_name}/{filename}',
                repo_type='model',
            )
            if Path(downloaded) != target_path:
                shutil.copyfile(downloaded, target_path)

print('Checkpoints ready.')


## Optional: use your own input

You can keep the sample folder, upload a video to `/content`, or mount Drive and point `INPUT_PATH` to a folder of images or a video file.


In [ ]:
MODEL_VARIANT = 'LoGeR'  # or 'LoGeR_star'
INPUT_PATH = 'data/examples/office'
DEVICE = 'auto'
START_FRAME = 0
END_FRAME = 50
STRIDE = 1
WINDOW_SIZE = 32
OVERLAP_SIZE = 3
SUBSAMPLE = 2
PORT = 8080
SHARE = True
SKIP_VISER = False

CONFIG_PATH = f'ckpts/{MODEL_VARIANT}/original_config.yaml'
MODEL_PATH = f'ckpts/{MODEL_VARIANT}/latest.pt'

print({
    'model': MODEL_VARIANT,
    'input': INPUT_PATH,
    'device': DEVICE,
    'share': SHARE,
    'skip_viser': SKIP_VISER,
})


In [ ]:
import os
import shlex

os.environ['XFORMERS_DISABLED'] = '1'

# Construct the command string
# Removed --device argument as it is not supported by demo_viser.py
cmd_str = f"python demo_viser.py --input {INPUT_PATH} --config {CONFIG_PATH} --model_name {MODEL_PATH} --start_frame {START_FRAME} --end_frame {END_FRAME} --stride {STRIDE} --window_size {WINDOW_SIZE} --overlap_size {OVERLAP_SIZE} --subsample {SUBSAMPLE} --port {PORT} --output_folder= results"

if SHARE:
    cmd_str += " --share"
if SKIP_VISER:
    cmd_str += " --skip_viser"

print(f'Running: {cmd_str}')
print('\nIf `--share` is enabled, watch the logs for the public `viser` URL.')

# Run with ! to see output directly
!{cmd_str}